In [7]:
# Cell 1：共用实验配置与环境记录

from pathlib import Path
import hashlib
import platform
import random
import sys
import time

import numpy as np


# 从 Notebook 当前目录向上定位仓库，所有输入、源码和输出均按工作区相对路径组织。
REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "workspaces" / "A3_性能").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("无法从当前目录向上定位 route_plan 仓库根目录。")
A3_ROOT = REPO_ROOT / "workspaces" / "A3_性能"
METHOD_DIR = A3_ROOT / "01_基准方法"
CODE_DIR = METHOD_DIR / "代码"
WORK_DIR = CODE_DIR
GRAPH_FILE = (A3_ROOT / "武汉路网数据" / "03_处理结果"
              / "2026-10-16_武汉路网复核" / "graph_v2.pkl")
QUERY_FILE = (A3_ROOT / "武汉路网数据" / "04_验证记录"
              / "2026-10-16_武汉路网数据画像" / "stratified_queries_100.json")

# Python 与 C++ 算法源文件
PY_BASELINE_FILE = REPO_ROOT / "workspaces" / "A2_算法" / "01_实现清单" / "astar.py"
PY_VARIANTS_FILE = CODE_DIR / "astar_variants.py"
CPP_SOURCE_FILE = CODE_DIR / "astar_cpp.cpp"

# 统一实验输出位置；本单元只定义路径，不创建目录
OUTPUT_DIR = A3_ROOT / "03_实验结果" / "武汉" / "Astar"
CPP_BUILD_DIR = CODE_DIR / ".build" / "astar_cpp"
CPP_DLL_FILE = CPP_BUILD_DIR / "astar_cpp.dll"

# 实验参数
RANDOM_SEED = 20261016
N_ROUNDS = 5
WARMUP_QUERY_COUNT = 5

EXPECTED_N_NODES = 91_638
EXPECTED_N_EDGES = 207_165
EXPECTED_N_QUERIES = 100


# 确认算法源文件存在，并记录哈希，便于之后复现实验版本
SOURCE_FILES = {
    "python_baseline": PY_BASELINE_FILE,
    "python_variants": PY_VARIANTS_FILE,
    "cpp": CPP_SOURCE_FILE,
}
missing_sources = [str(path) for path in SOURCE_FILES.values() if not path.is_file()]
if missing_sources:
    raise FileNotFoundError("找不到以下算法源文件：\n" + "\n".join(missing_sources))

SOURCE_SHA256 = {
    name: hashlib.sha256(path.read_bytes()).hexdigest()
    for name, path in SOURCE_FILES.items()
}

ENVIRONMENT = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "numpy": np.__version__,
}

print("实验工作目录：", WORK_DIR)
print("武汉路网：", GRAPH_FILE)
print("固定查询集：", QUERY_FILE)
print(f"实验参数：{N_ROUNDS} 轮，预热 {WARMUP_QUERY_COUNT} 个查询，随机种子 {RANDOM_SEED}")
print("预期规模："
      f"{EXPECTED_N_NODES:,} 个节点，"
      f"{EXPECTED_N_EDGES:,} 条有向边，"
      f"{EXPECTED_N_QUERIES} 个查询")
print("运行环境：", ENVIRONMENT)
print("算法源文件 SHA-256：")
for name, digest in SOURCE_SHA256.items():
    print(f"  {name}: {digest}")

实验工作目录： D:\pycharm\projects\route_plan\A3_临时工作区\06_CHATGPT\基础代码与优化测试结果
武汉路网： D:\pycharm\projects\route_plan\workspaces\A3_性能\武汉路网数据\03_处理结果\2026-10-16_武汉路网复核\graph_v2.pkl
固定查询集： D:\pycharm\projects\route_plan\workspaces\A3_性能\武汉路网数据\04_验证记录\2026-10-16_武汉路网数据画像\stratified_queries_100.json
实验参数：5 轮，预热 5 个查询，随机种子 20261016
预期规模：91,638 个节点，207,165 条有向边，100 个查询
运行环境： {'python': '3.11.17', 'platform': 'Windows-10-10.0.26200-SP0', 'machine': 'AMD64', 'numpy': '2.4.6'}
算法源文件 SHA-256：
  python_baseline: 83ad98d5f334ded67fbbad56131e23991d0be3bf15a2deca9186881c40da8164
  python_variants: 6d81618eca7662705fe4527fd5ab1d65df5cf4e07db1c35c070bb554428c0495
  cpp: 5f7c6519a01bc92ca64578a80e94e1a66e95f2663f00ecfabce247bb96500ffc


In [8]:
# Cell 2：共同加载路网与查询集，准备 Python/C++ 共用输入

import json
import pickle


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


# 加载原始路网
t0 = time.perf_counter_ns()
with GRAPH_FILE.open("rb") as file:
    GRAPH_DATA = pickle.load(file)
GRAPH_LOAD_NS = time.perf_counter_ns() - t0

# 规范为连续数组：Python 和 C++ 共用同一份 CSR 数据
t0 = time.perf_counter_ns()

N_NODES = int(GRAPH_DATA["n_nodes"])
INDPTR = np.ascontiguousarray(GRAPH_DATA["indptr"], dtype=np.int64)
INDICES = np.ascontiguousarray(GRAPH_DATA["indices"], dtype=np.int64)
WEIGHTS = np.ascontiguousarray(GRAPH_DATA["weights"], dtype=np.float64)
EDGE_IDS = np.ascontiguousarray(GRAPH_DATA["edge_ids"], dtype=np.int64)
LON = np.ascontiguousarray(GRAPH_DATA["lon"], dtype=np.float64)
LAT = np.ascontiguousarray(GRAPH_DATA["lat"], dtype=np.float64)

N_EDGES = len(INDICES)

if N_NODES != EXPECTED_N_NODES or N_EDGES != EXPECTED_N_EDGES:
    raise ValueError(
        f"路网规模与实验预期不符：节点 {N_NODES:,}，边 {N_EDGES:,}"
    )
if INDPTR.shape != (N_NODES + 1,) or INDPTR[0] != 0 or INDPTR[-1] != N_EDGES:
    raise ValueError("CSR indptr 长度或首尾位置不正确")
if np.any(INDPTR[1:] < INDPTR[:-1]):
    raise ValueError("CSR indptr 必须单调不减")
if any(array.shape != (N_EDGES,) for array in (WEIGHTS, EDGE_IDS)):
    raise ValueError("indices、weights、edge_ids 的长度不一致")
if any(array.shape != (N_NODES,) for array in (LON, LAT)):
    raise ValueError("经纬度数组长度与节点数不一致")
if np.any(INDICES < 0) or np.any(INDICES >= N_NODES):
    raise ValueError("邻接节点编号超出范围")
if not np.isfinite(WEIGHTS).all() or np.any(WEIGHTS < 0):
    raise ValueError("边权必须是有限的非负数")
if not np.isfinite(LON).all() or not np.isfinite(LAT).all():
    raise ValueError("节点经纬度包含非有限值")

GRAPH_POLICY = GRAPH_DATA["policy"]
CSR_GRAPH = {
    "n_nodes": N_NODES,
    "n_edges": N_EDGES,
    "indptr": INDPTR,
    "indices": INDICES,
    "weights": WEIGHTS,
    "edge_ids": EDGE_IDS,
    "lon": LON,
    "lat": LAT,
}
CSR_PREP_NS = time.perf_counter_ns() - t0

# 转为 Python A* 使用的 {node: [(neighbor, weight, edge_id), ...]} 结构
t0 = time.perf_counter_ns()
PY_GRAPH = {
    node: list(zip(
        INDICES[INDPTR[node]:INDPTR[node + 1]].tolist(),
        WEIGHTS[INDPTR[node]:INDPTR[node + 1]].tolist(),
        EDGE_IDS[INDPTR[node]:INDPTR[node + 1]].tolist(),
    ))
    for node in range(N_NODES)
}
POSITIONS = {
    node: (float(LON[node]), float(LAT[node]))
    for node in range(N_NODES)
}
PY_ADJ_BUILD_NS = time.perf_counter_ns() - t0

# 加载固定查询集，Python 与 C++ 后续使用相同的端点顺序
t0 = time.perf_counter_ns()
with QUERY_FILE.open("r", encoding="utf-8") as file:
    QUERY_DATA = json.load(file)

if QUERY_DATA["seed"] != RANDOM_SEED:
    raise ValueError(f"查询集随机种子不符：{QUERY_DATA['seed']}")
if QUERY_DATA["graph_policy"] != GRAPH_POLICY:
    raise ValueError("查询集与路网的 graph_policy 不一致")

FIXED_QUERIES = QUERY_DATA["queries"]
if len(FIXED_QUERIES) != EXPECTED_N_QUERIES:
    raise ValueError(f"查询数不符：预期 {EXPECTED_N_QUERIES}，实际 {len(FIXED_QUERIES)}")

QUERY_PAIRS = [
    (int(query["source"]), int(query["target"]))
    for query in FIXED_QUERIES
]
if any(
    source < 0 or source >= N_NODES or target < 0 or target >= N_NODES
    for source, target in QUERY_PAIRS
):
    raise ValueError("查询集包含超出路网范围的节点编号")

QUERY_LOAD_NS = time.perf_counter_ns() - t0

GRAPH_SHA256 = sha256_file(GRAPH_FILE)
QUERY_SHA256 = sha256_file(QUERY_FILE)

print(f"路网：{N_NODES:,} 个节点，{N_EDGES:,} 条有向边")
print(f"查询：{len(QUERY_PAIRS)} 对；graph_policy={GRAPH_POLICY}")
print(f"图文件 SHA-256：{GRAPH_SHA256}")
print(f"查询集 SHA-256：{QUERY_SHA256}")
print(
    "准备耗时："
    f"读图 {GRAPH_LOAD_NS / 1e6:.1f} ms，"
    f"CSR 检查/规范化 {CSR_PREP_NS / 1e6:.1f} ms，"
    f"Python 邻接表转换 {PY_ADJ_BUILD_NS / 1e6:.1f} ms，"
    f"读查询 {QUERY_LOAD_NS / 1e6:.1f} ms"
)

路网：91,638 个节点，207,165 条有向边
查询：100 对；graph_policy=wuhan-static-public-way-distance-v2.1
图文件 SHA-256：fe7492eaadb4c0d058f8d00c78aa79ee9d6597ac23d4b3280b9c0d09faaf73fa
查询集 SHA-256：aa5c728689ec18f6ef0c20e8e11fdbc8fb17063cb871abd551021bb86ba6950c
准备耗时：读图 49.5 ms，CSR 检查/规范化 6.8 ms，Python 邻接表转换 266.7 ms，读查询 0.8 ms


In [10]:
# Cell 3：加载 Python 算法，并构建/加载 C++ 库、创建 C++ 图对象

import ctypes as ct
import importlib.util
import os
import shutil
import subprocess


# 加载已有 Python 算法文件，不在 Notebook 中复制算法实现
def load_algorithm_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"无法加载算法模块：{file_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


ASTAR_BASELINE = load_algorithm_module("astar_baseline", PY_BASELINE_FILE)
ASTAR_VARIANTS = load_algorithm_module("astar_variants", PY_VARIANTS_FILE)


# 若重跑本格，先释放上次创建的 C++ 图对象
_old_cpp_lib = globals().get("CPP_LIB")
_old_cpp_handle = globals().get("CPP_GRAPH_HANDLE")
if _old_cpp_lib is not None and _old_cpp_handle:
    _old_cpp_lib.astar_graph_destroy(_old_cpp_handle)
CPP_GRAPH_HANDLE = None


# DLL 文件名绑定源码哈希，源码更新时不会覆盖仍被 Notebook 加载的旧 DLL
CPP_SOURCE_SHA256 = sha256_file(CPP_SOURCE_FILE)
SOURCE_SHA256["cpp"] = CPP_SOURCE_SHA256

CPP_BUILD_FLAGS = ("/std:c++17", "/O2", "/EHsc", "/LD", "/DNDEBUG")
CPP_BUILD_KEY = hashlib.sha256(
    (CPP_SOURCE_SHA256 + "|".join(CPP_BUILD_FLAGS)).encode("utf-8")
).hexdigest()[:12]

CPP_BUILD_DIR.mkdir(parents=True, exist_ok=True)
CPP_DLL_FILE = CPP_BUILD_DIR / f"astar_cpp_{CPP_BUILD_KEY}.dll"
CPP_IMPLIB_FILE = CPP_BUILD_DIR / f"astar_cpp_{CPP_BUILD_KEY}.lib"

CPP_BUILD_REUSED = CPP_DLL_FILE.is_file()
CPP_BUILD_NS = 0

if not CPP_BUILD_REUSED:
    cl_exe = shutil.which("cl.exe")
    vswhere = Path(
        os.environ.get("ProgramFiles(x86)", r"C:\Program Files (x86)")
    ) / "Microsoft Visual Studio" / "Installer" / "vswhere.exe"

    if vswhere.is_file():
        vs_result = subprocess.run(
            [
                str(vswhere), "-latest", "-products", "*",
                "-requires", "Microsoft.VisualStudio.Component.VC.Tools.x86.x64",
                "-property", "installationPath",
            ],
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        install_path = vs_result.stdout.strip()
    else:
        install_path = ""

    vcvars64 = (
        Path(install_path) / "VC" / "Auxiliary" / "Build" / "vcvars64.bat"
        if install_path else None
    )

    compile_args = [
        "/nologo", *CPP_BUILD_FLAGS, str(CPP_SOURCE_FILE),
        "/link", f"/OUT:{CPP_DLL_FILE}", f"/IMPLIB:{CPP_IMPLIB_FILE}",
    ]

    build_start_ns = time.perf_counter_ns()

    if vcvars64 is not None and vcvars64.is_file():
        compile_command = (
            f'call "{vcvars64}" && cl /nologo /std:c++17 /O2 /EHsc /LD /DNDEBUG '
            f'"{CPP_SOURCE_FILE}" /link '
            f'/OUT:"{CPP_DLL_FILE}" /IMPLIB:"{CPP_IMPLIB_FILE}"'
        )
        build_result = subprocess.run(
            compile_command,
            shell=True,  # 这里需要 CMD 的 call 和 && 语义
            cwd=CPP_BUILD_DIR,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        CPP_COMPILER = "MSVC x64 via vcvars64.bat"
    elif cl_exe is not None:
        build_result = subprocess.run(
            [cl_exe, *compile_args],
            cwd=CPP_BUILD_DIR,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        CPP_COMPILER = cl_exe
    else:
        raise RuntimeError(
            "未找到可用的 MSVC x64 编译环境。请确认已安装 Visual Studio C++ "
            "生成工具，或从已配置 MSVC 的终端启动 Notebook 内核。"
        )

    CPP_BUILD_NS = time.perf_counter_ns() - build_start_ns
    if build_result.returncode != 0 or not CPP_DLL_FILE.is_file():
        raise RuntimeError(
            "A* C++ DLL 构建失败：\n"
            + build_result.stdout[-3000:]
            + build_result.stderr[-3000:]
        )
else:
    CPP_COMPILER = "复用与当前源码哈希匹配的 DLL"


# 绑定 C++ ABI
if ct.sizeof(ct.c_void_p) != 8:
    raise RuntimeError("当前 C++ DLL 需要 64 位 Python 内核")

try:
    CPP_LIB = ct.CDLL(str(CPP_DLL_FILE))
except OSError as exc:
    raise RuntimeError(
        f"DLL 加载失败：{CPP_DLL_FILE}\n"
        "请检查 Python/DLL 位数及 MSVC 运行库。"
    ) from exc

CPP_I64_P = ct.POINTER(ct.c_int64)
CPP_DOUBLE_P = ct.POINTER(ct.c_double)
CPP_U64_P = ct.POINTER(ct.c_uint64)

CPP_STATUS_NAMES = {
    0: "OK",
    1: "INVALID_ARGUMENT",
    2: "INVALID_GRAPH",
    3: "INVALID_QUERY",
    4: "BUFFER_TOO_SMALL",
    5: "PREPARATION_REQUIRED",
    6: "INTERNAL_ERROR",
}

CPP_VERSION_IDS = {
    "L0": 0,
    "L0-CachedScale": 1,
    "L1": 2,
    "L2": 3,
    "L3": 4,
    "L4": 5,
    "L5": 6,
    "B1": 7,
}
CPP_HEURISTIC_IDS = {
    "H0": 0,
    "PLANE": 1,
    "ALT": 2,
    "MAX": 3,
    "MAX_PLANE": 4,
}

CPP_LIB.astar_graph_create.argtypes = [
    ct.c_int64, ct.c_int64,
    CPP_I64_P, CPP_I64_P, CPP_DOUBLE_P, CPP_I64_P,
    CPP_DOUBLE_P, CPP_DOUBLE_P,
    ct.POINTER(ct.c_int32),
]
CPP_LIB.astar_graph_create.restype = ct.c_void_p
CPP_LIB.astar_graph_destroy.argtypes = [ct.c_void_p]
CPP_LIB.astar_graph_destroy.restype = None


# C++ 图对象在创建时复制输入数组；建图时间单独记录，不计入查询时间
CPP_GRAPH_ERROR = ct.c_int32()
cpp_graph_start_ns = time.perf_counter_ns()

CPP_GRAPH_HANDLE = CPP_LIB.astar_graph_create(
    N_NODES,
    N_EDGES,
    INDPTR.ctypes.data_as(CPP_I64_P),
    INDICES.ctypes.data_as(CPP_I64_P),
    WEIGHTS.ctypes.data_as(CPP_DOUBLE_P),
    EDGE_IDS.ctypes.data_as(CPP_I64_P),
    LON.ctypes.data_as(CPP_DOUBLE_P),
    LAT.ctypes.data_as(CPP_DOUBLE_P),
    ct.byref(CPP_GRAPH_ERROR),
)

CPP_GRAPH_CREATE_NS = time.perf_counter_ns() - cpp_graph_start_ns

if not CPP_GRAPH_HANDLE or CPP_GRAPH_ERROR.value != 0:
    status = CPP_STATUS_NAMES.get(
        CPP_GRAPH_ERROR.value, f"UNKNOWN_STATUS_{CPP_GRAPH_ERROR.value}"
    )
    raise RuntimeError(f"C++ 建图失败：{status}")

CPP_DLL_SHA256 = sha256_file(CPP_DLL_FILE)

print("Python 模块：A2 基线与 astar_variants.py 已加载")
print(f"C++ 编译器：{CPP_COMPILER}")
print(f"C++ DLL：{CPP_DLL_FILE}")
print(f"DLL SHA-256：{CPP_DLL_SHA256}")
print(f"本次构建耗时：{CPP_BUILD_NS / 1e6:.1f} ms；复用已有 DLL：{CPP_BUILD_REUSED}")
print(f"C++ 建图耗时：{CPP_GRAPH_CREATE_NS / 1e6:.1f} ms")
print(f"C++ 图对象就绪：{N_NODES:,} 个节点，{N_EDGES:,} 条有向边")

Python 模块：A2 基线与 astar_variants.py 已加载
C++ 编译器：MSVC x64 via vcvars64.bat
C++ DLL：D:\pycharm\projects\route_plan\A3_临时工作区\06_CHATGPT\基础代码与优化测试结果\实验输出\astar_python_cpp\cpp\astar_cpp_c9a2280dbf61.dll
DLL SHA-256：e1eabda056b56616c12d75ed1c77265516d0349552fa40e52b3105e5f68dd80f
本次构建耗时：3601.3 ms；复用已有 DLL：False
C++ 建图耗时：13.8 ms
C++ 图对象就绪：91,638 个节点，207,165 条有向边


In [11]:
# Cell 4：启发式与 ALT 预处理，定义 Python/C++ 共用测试接口

from collections import Counter

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components


# Python 一次性准备
PY_PREP_NS = {}

t0 = time.perf_counter_ns()
if ASTAR_VARIANTS.prevalidate_graph(PY_GRAPH, POSITIONS) != N_NODES:
    raise ValueError("Python 图预检查返回的节点数不符")
PY_PREP_NS["graph_validation"] = time.perf_counter_ns() - t0

t0 = time.perf_counter_ns()
H0_SCALE = ASTAR_VARIANTS.estimate_heuristic_scale(PY_GRAPH, POSITIONS)
PY_PREP_NS["h0_scale"] = time.perf_counter_ns() - t0

t0 = time.perf_counter_ns()
PY_PROJECTED = ASTAR_VARIANTS.project_coordinates(POSITIONS)
PY_PLANE_SCALE = ASTAR_VARIANTS.estimate_plane_scale(PY_GRAPH, PY_PROJECTED)
PY_PREP_NS["plane"] = time.perf_counter_ns() - t0

# 原始 CSR 数组已在 Cell 2 检查并规范化，直接供 L4/L5 复用。
PY_CSR = (INDPTR, INDICES, WEIGHTS, EDGE_IDS)

t0 = time.perf_counter_ns()
PY_REVERSE_GRAPH = ASTAR_VARIANTS.build_reverse_graph(PY_GRAPH)
PY_PREP_NS["reverse_graph"] = time.perf_counter_ns() - t0


# 计算强连通分量，优先从覆盖固定查询最多的分量中选地标
SCC_MATRIX = csr_matrix(
    (np.ones(N_EDGES, dtype=np.uint8), INDICES, INDPTR),
    shape=(N_NODES, N_NODES),
)
SCC_COUNT, SCC_LABELS = connected_components(
    SCC_MATRIX,
    directed=True,
    connection="strong",
    return_labels=True,
)
SCC_SIZES = np.bincount(SCC_LABELS, minlength=SCC_COUNT)

SAME_SCC_QUERY_COUNTS = Counter(
    int(SCC_LABELS[source])
    for source, target in QUERY_PAIRS
    if SCC_LABELS[source] == SCC_LABELS[target]
)

if SAME_SCC_QUERY_COUNTS:
    LANDMARK_COMPONENT = min(
        SAME_SCC_QUERY_COUNTS,
        key=lambda component: (
            -SAME_SCC_QUERY_COUNTS[component],
            -SCC_SIZES[component],
            component,
        ),
    )
else:
    LANDMARK_COMPONENT = int(np.argmax(SCC_SIZES))

LANDMARK_CANDIDATES = np.flatnonzero(
    SCC_LABELS == LANDMARK_COMPONENT
).tolist()

if len(LANDMARK_CANDIDATES) < 16:
    raise ValueError("所选强连通分量不足 16 个节点，无法构造 ALT-16")

t0 = time.perf_counter_ns()
LANDMARK_IDS = ASTAR_VARIANTS.select_landmarks(
    POSITIONS,
    LANDMARK_CANDIDATES,
    count_requested=16,
)
PY_PREP_NS["landmark_selection"] = time.perf_counter_ns() - t0

t0 = time.perf_counter_ns()
PY_ALT_TABLES = ASTAR_VARIANTS.prepare_landmarks(PY_GRAPH, LANDMARK_IDS)
PY_PREP_NS["alt_tables"] = time.perf_counter_ns() - t0


# 绑定 C++ 预处理与查询接口
CPP_LIB.astar_prepare_geometry.argtypes = [
    ct.c_void_p, CPP_DOUBLE_P, CPP_DOUBLE_P, CPP_U64_P
]
CPP_LIB.astar_prepare_geometry.restype = ct.c_int32

CPP_LIB.astar_prepare_reverse.argtypes = [ct.c_void_p, CPP_U64_P]
CPP_LIB.astar_prepare_reverse.restype = ct.c_int32

CPP_LIB.astar_prepare_alt.argtypes = [
    ct.c_void_p, ct.c_int32, CPP_I64_P, CPP_U64_P
]
CPP_LIB.astar_prepare_alt.restype = ct.c_int32

CPP_LIB.astar_run.argtypes = [
    ct.c_void_p,
    ct.c_int32, ct.c_int32, ct.c_int32,
    ct.c_int64, ct.c_int64,
    CPP_I64_P, ct.c_int64,
    CPP_I64_P, ct.c_int64,
    ct.POINTER(ct.c_int64), ct.POINTER(ct.c_int64),
    CPP_DOUBLE_P,
    CPP_U64_P, CPP_U64_P, CPP_U64_P, CPP_U64_P, CPP_U64_P,
]
CPP_LIB.astar_run.restype = ct.c_int32


def check_cpp_status(status, operation):
    if status != 0:
        name = CPP_STATUS_NAMES.get(status, f"UNKNOWN_STATUS_{status}")
        raise RuntimeError(f"C++ {operation} 失败：{name}")


# 几何比例、反向图和 ALT 表都按图预处理一次；内部耗时单独记录
CPP_H0_SCALE = ct.c_double()
CPP_PLANE_SCALE = ct.c_double()
_CPP_ELAPSED = ct.c_uint64()

t0 = time.perf_counter_ns()
status = CPP_LIB.astar_prepare_geometry(
    CPP_GRAPH_HANDLE,
    ct.byref(CPP_H0_SCALE),
    ct.byref(CPP_PLANE_SCALE),
    ct.byref(_CPP_ELAPSED),
)
CPP_GEOMETRY_OUTER_NS = time.perf_counter_ns() - t0
check_cpp_status(status, "几何预处理")
CPP_GEOMETRY_KERNEL_NS = int(_CPP_ELAPSED.value)

t0 = time.perf_counter_ns()
status = CPP_LIB.astar_prepare_reverse(
    CPP_GRAPH_HANDLE, ct.byref(_CPP_ELAPSED)
)
CPP_REVERSE_OUTER_NS = time.perf_counter_ns() - t0
check_cpp_status(status, "反向图预处理")
CPP_REVERSE_KERNEL_NS = int(_CPP_ELAPSED.value)

CPP_LANDMARK_IDS = np.ascontiguousarray(LANDMARK_IDS, dtype=np.int64)
t0 = time.perf_counter_ns()
status = CPP_LIB.astar_prepare_alt(
    CPP_GRAPH_HANDLE,
    len(CPP_LANDMARK_IDS),
    CPP_LANDMARK_IDS.ctypes.data_as(CPP_I64_P),
    ct.byref(_CPP_ELAPSED),
)
CPP_ALT_OUTER_NS = time.perf_counter_ns() - t0
check_cpp_status(status, "ALT 预处理")
CPP_ALT_KERNEL_NS = int(_CPP_ELAPSED.value)


# 缓冲区容量按简单路径最多 n 个节点、n-1 条边设置
CPP_NODE_BUFFER = np.empty(N_NODES, dtype=np.int64)
CPP_EDGE_BUFFER = np.empty(N_NODES, dtype=np.int64)


def run_cpp_case(case, source, target):
    node_count = ct.c_int64()
    edge_count = ct.c_int64()
    distance = ct.c_double()
    expanded = ct.c_uint64()
    pushed = ct.c_uint64()
    scanned = ct.c_uint64()
    heuristic_evaluations = ct.c_uint64()
    elapsed_ns = ct.c_uint64()

    status = CPP_LIB.astar_run(
        CPP_GRAPH_HANDLE,
        CPP_VERSION_IDS[case["version"]],
        CPP_HEURISTIC_IDS[case["heuristic"]],
        case["k"],
        int(source),
        int(target),
        CPP_NODE_BUFFER.ctypes.data_as(CPP_I64_P),
        CPP_NODE_BUFFER.size,
        CPP_EDGE_BUFFER.ctypes.data_as(CPP_I64_P),
        CPP_EDGE_BUFFER.size,
        ct.byref(node_count),
        ct.byref(edge_count),
        ct.byref(distance),
        ct.byref(expanded),
        ct.byref(pushed),
        ct.byref(scanned),
        ct.byref(heuristic_evaluations),
        ct.byref(elapsed_ns),
    )
    check_cpp_status(status, f'{case["name"]} 查询')

    if not (
        0 <= node_count.value <= CPP_NODE_BUFFER.size
        and 0 <= edge_count.value <= CPP_EDGE_BUFFER.size
    ):
        raise RuntimeError("C++ 返回的路径长度超出输出缓冲区")

    return {
        "nodes": CPP_NODE_BUFFER[:node_count.value].tolist(),
        "edge_ids": CPP_EDGE_BUFFER[:edge_count.value].tolist(),
        "distance_m": float(distance.value),
        "expanded_nodes": int(expanded.value),
        "pushed_count": int(pushed.value),
        "scanned_edges": int(scanned.value),
        "heuristic_evaluations": int(heuristic_evaluations.value),
        "kernel_ns": int(elapsed_ns.value),
    }


CASE_SPECS = [
    {"name": "L0", "version": "L0", "heuristic": "H0", "k": 0},
    {
        "name": "L0-CachedScale",
        "version": "L0-CachedScale",
        "heuristic": "H0",
        "k": 0,
    },
] + [
    {"name": layer, "version": layer, "heuristic": "H0", "k": 0}
    for layer in ("L1", "L2", "L3", "L4", "L5")
] + [
    {"name": "L4-PLANE", "version": "L4", "heuristic": "PLANE", "k": 0},
] + [
    {"name": f"L4-ALT{k}", "version": "L4", "heuristic": "ALT", "k": k}
    for k in (4, 8, 16)
] + [
    {"name": f"L4-MAX{k}", "version": "L4", "heuristic": "MAX", "k": k}
    for k in (4, 8, 16)
] + [
    {"name": "B1-H0", "version": "B1", "heuristic": "H0", "k": 0},
]
CASE_ORDER = tuple(case["name"] for case in CASE_SPECS)


def run_python_case(case, source, target, *, collect_stats=False):
    stats = {} if collect_stats else None

    if case["name"] == "L0":
        result = ASTAR_BASELINE.astar(
            PY_GRAPH, POSITIONS, int(source), int(target)
        )
    elif case["version"] == "B1":
        result = ASTAR_VARIANTS.bidirectional_astar(
            PY_GRAPH,
            PY_REVERSE_GRAPH,
            POSITIONS,
            int(source),
            int(target),
            heuristic=case["heuristic"],
            heuristic_scale=H0_SCALE,
            projected=PY_PROJECTED,
            plane_scale=PY_PLANE_SCALE,
            alt=PY_ALT_TABLES,
            landmark_count=case["k"],
            stats=stats,
        )
    else:
        result = ASTAR_VARIANTS.run_astar(
            case["version"],
            PY_GRAPH,
            POSITIONS,
            int(source),
            int(target),
            heuristic=case["heuristic"],
            heuristic_scale=H0_SCALE,
            projected=PY_PROJECTED,
            plane_scale=PY_PLANE_SCALE,
            csr=PY_CSR,
            alt=PY_ALT_TABLES,
            landmark_count=case["k"],
            stats=stats,
        )

    nodes, edge_ids, distance, expanded = result
    return {
        "nodes": nodes,
        "edge_ids": edge_ids,
        "distance_m": float(distance),
        "expanded_nodes": int(expanded),
        "pushed_count": None if stats is None else stats.get("pushed_count"),
        "scanned_edges": None if stats is None else stats.get("scanned_edges"),
        "heuristic_evaluations": (
            None if stats is None else stats.get("heuristic_evaluations")
        ),
    }


print(
    f"SCC 数：{SCC_COUNT:,}；同属一个 SCC 的查询："
    f"{sum(SAME_SCC_QUERY_COUNTS.values())}/{len(QUERY_PAIRS)}"
)
print(
    f"ALT 地标分量：{LANDMARK_COMPONENT}，"
    f"分量节点数：{len(LANDMARK_CANDIDATES):,}，"
    f"地标 ID：{LANDMARK_IDS}"
)
print(
    f"H0 scale：Python={H0_SCALE:.12g}，"
    f"C++={CPP_H0_SCALE.value:.12g}；"
    f"平面 scale：Python={PY_PLANE_SCALE:.12g}，"
    f"C++={CPP_PLANE_SCALE.value:.12g}"
)
print(
    f"预处理完成：Python ALT {PY_PREP_NS['alt_tables'] / 1e9:.2f} s；"
    f"C++ ALT 内部 {CPP_ALT_KERNEL_NS / 1e9:.2f} s；"
    f"共 {len(CASE_ORDER)} 个测试配置"
)

SCC 数：1,311；同属一个 SCC 的查询：100/100
ALT 地标分量：399，分量节点数：89,806，地标 ID：[80958, 41381, 61502, 90950, 16746, 21829, 21256, 16433, 81239, 18698, 13869, 20586, 6363, 12080, 41389, 8050]
H0 scale：Python=0.999999997678，C++=0.999999999999；平面 scale：Python=0.993053607524，C++=0.993053607524
预处理完成：Python ALT 2.52 s；C++ ALT 内部 0.26 s；共 15 个测试配置


In [12]:
# Cell 5：正确性验证
# 对固定 100 个查询，逐一检查 15 个配置的 Python/C++ 结果；本格不计时、不写文件。

import math


DISTANCE_ATOL_M = 1e-6
DISTANCE_RTOL = 1e-10


def validate_astar_result(result, source, target, reference_distance, label):
    nodes = result["nodes"]
    edge_ids = result["edge_ids"]
    reported_distance = result["distance_m"]

    if not nodes or nodes[0] != source or nodes[-1] != target:
        raise AssertionError(f"{label}：路径起点或终点不正确")
    if len(edge_ids) != len(nodes) - 1:
        raise AssertionError(f"{label}：节点数与 edge_id 数量不匹配")
    if not math.isfinite(reported_distance) or reported_distance < 0:
        raise AssertionError(f"{label}：返回距离不是有限非负数")

    path_weight = 0.0
    for u, v, edge_id in zip(nodes, nodes[1:], edge_ids):
        matches = [
            float(weight)
            for neighbor, weight, actual_edge_id in PY_GRAPH[u]
            if neighbor == v and actual_edge_id == edge_id
        ]
        if not matches:
            raise AssertionError(
                f"{label}：路径边不存在或 edge_id 不匹配：{u}->{v}, edge_id={edge_id}"
            )
        path_weight += matches[0]

    if not math.isclose(
        path_weight, reported_distance,
        rel_tol=DISTANCE_RTOL, abs_tol=DISTANCE_ATOL_M
    ):
        raise AssertionError(
            f"{label}：路径边权和 {path_weight:.12f} "
            f"与返回距离 {reported_distance:.12f} 不一致"
        )

    if reference_distance is not None and not math.isclose(
        reported_distance, reference_distance,
        rel_tol=DISTANCE_RTOL, abs_tol=DISTANCE_ATOL_M
    ):
        raise AssertionError(
            f"{label}：距离 {reported_distance:.12f} "
            f"与 A2 基线 {reference_distance:.12f} 不一致"
        )


CHECKED_RUNS = 0

for query_index, (source, target) in enumerate(QUERY_PAIRS, start=1):
    reference_distance = None

    for case in CASE_SPECS:
        case_name = case["name"]

        py_result = run_python_case(case, source, target)
        if case_name == "L0":
            # A2 收件箱版 Python A* 作为该查询的最短距离参照。
            reference_distance = py_result["distance_m"]

        validate_astar_result(
            py_result,
            source,
            target,
            None if case_name == "L0" else reference_distance,
            f"Python/{case_name}/查询{query_index}",
        )
        CHECKED_RUNS += 1

        cpp_result = run_cpp_case(case, source, target)
        validate_astar_result(
            cpp_result,
            source,
            target,
            reference_distance,
            f"C++/{case_name}/查询{query_index}",
        )
        CHECKED_RUNS += 1

H0_SCALE_RELATIVE_GAP = abs(H0_SCALE - CPP_H0_SCALE.value) / max(
    abs(H0_SCALE), abs(CPP_H0_SCALE.value), 1.0
)

print(
    f"正确性验证通过：{len(QUERY_PAIRS)} 个查询 × "
    f"{len(CASE_ORDER)} 个配置 × 2 种语言 = {CHECKED_RUNS:,} 次算法运行"
)
print("检查项：路径端点、逐段邻接与 edge_id、路径边权和、相对 A2 基线的最短距离")
print(
    f"H0 scale 相对差异：{H0_SCALE_RELATIVE_GAP:.3e}；"
    "保留两种语言各自计算的保守 scale"
)
print("说明：等长最优路径可能经过不同路段，因此不要求 Python/C++ 的 edge_id 序列完全相同。")

正确性验证通过：100 个查询 × 15 个配置 × 2 种语言 = 3,000 次算法运行
检查项：路径端点、逐段邻接与 edge_id、路径边权和、相对 A2 基线的最短距离
H0 scale 相对差异：2.321e-09；保留两种语言各自计算的保守 scale
说明：等长最优路径可能经过不同路段，因此不要求 Python/C++ 的 edge_id 序列完全相同。


In [13]:
# Cell 6：配对计时、汇总并保存结果
# 外层耗时用于 Python/C++ 对比；C++ 内核耗时单独记录。

import pandas as pd
from IPython.display import display


EXPECTED_CHECKED_RUNS = len(QUERY_PAIRS) * len(CASE_ORDER) * 2
if CHECKED_RUNS != EXPECTED_CHECKED_RUNS:
    raise RuntimeError(
        f"Cell 5 正确性验证未完整通过：预期 {EXPECTED_CHECKED_RUNS} 次，"
        f"实际 {CHECKED_RUNS} 次"
    )

BENCHMARK_RNG = random.Random(RANDOM_SEED)
BENCHMARK_ITEMS = [
    (case, language)
    for case in CASE_SPECS
    for language in ("Python", "C++")
]

# 每个配置、每种语言都先运行相同的前 N 个固定查询，不计入正式结果。
WARMUP_ITEMS = BENCHMARK_ITEMS.copy()
BENCHMARK_RNG.shuffle(WARMUP_ITEMS)

for query_index, (source, target) in enumerate(
    QUERY_PAIRS[:WARMUP_QUERY_COUNT], start=1
):
    for case, language in WARMUP_ITEMS:
        if language == "Python":
            run_python_case(case, source, target)
        else:
            run_cpp_case(case, source, target)

# 正式计时：同一查询、同一轮内随机打乱配置和语言的执行顺序。
TIMING_RECORDS = []

for round_index in range(1, N_ROUNDS + 1):
    for query_index, (source, target) in enumerate(QUERY_PAIRS, start=1):
        round_items = BENCHMARK_ITEMS.copy()
        BENCHMARK_RNG.shuffle(round_items)

        for order_index, (case, language) in enumerate(round_items, start=1):
            t0 = time.perf_counter_ns()

            if language == "Python":
                result = run_python_case(case, source, target)
                kernel_ns = None
            else:
                result = run_cpp_case(case, source, target)
                kernel_ns = result["kernel_ns"]

            outer_ns = time.perf_counter_ns() - t0

            TIMING_RECORDS.append({
                "round": round_index,
                "query_index": query_index,
                "source": source,
                "target": target,
                "order_index": order_index,
                "case": case["name"],
                "version": case["version"],
                "heuristic": case["heuristic"],
                "landmark_count": case["k"],
                "language": language,
                "outer_ns": outer_ns,
                "outer_ms": outer_ns / 1e6,
                "kernel_ns": kernel_ns,
                "kernel_ms": (
                    None if kernel_ns is None else kernel_ns / 1e6
                ),
                "expanded_nodes": result["expanded_nodes"],
                "pushed_count": result["pushed_count"],
                "scanned_edges": result["scanned_edges"],
                "heuristic_evaluations": result["heuristic_evaluations"],
            })

EXPECTED_TIMING_RECORDS = N_ROUNDS * len(QUERY_PAIRS) * len(CASE_ORDER) * 2
if len(TIMING_RECORDS) != EXPECTED_TIMING_RECORDS:
    raise RuntimeError(
        f"计时记录数不符：预期 {EXPECTED_TIMING_RECORDS}，"
        f"实际 {len(TIMING_RECORDS)}"
    )

# 配对索引：(配置、语言、轮次、查询编号) -> 计时记录
TIMING_LOOKUP = {
    (row["case"], row["language"], row["round"], row["query_index"]): row
    for row in TIMING_RECORDS
}

SUMMARY_ROWS = []

for case_name in CASE_ORDER:
    py_rows = [
        row for row in TIMING_RECORDS
        if row["case"] == case_name and row["language"] == "Python"
    ]
    cpp_rows = [
        row for row in TIMING_RECORDS
        if row["case"] == case_name and row["language"] == "C++"
    ]

    py_ms = np.asarray([row["outer_ms"] for row in py_rows], dtype=np.float64)
    cpp_ms = np.asarray([row["outer_ms"] for row in cpp_rows], dtype=np.float64)
    cpp_kernel_ms = np.asarray(
        [row["kernel_ms"] for row in cpp_rows], dtype=np.float64
    )

    py_q1, py_median, py_q3 = np.percentile(py_ms, [25, 50, 75])
    cpp_q1, cpp_median, cpp_q3 = np.percentile(cpp_ms, [25, 50, 75])
    kernel_median = float(np.median(cpp_kernel_ms))

    paired_ratios = np.asarray([
        TIMING_LOOKUP[
            (case_name, "Python", round_index, query_index)
        ]["outer_ns"]
        / TIMING_LOOKUP[
            (case_name, "C++", round_index, query_index)
        ]["outer_ns"]
        for round_index in range(1, N_ROUNDS + 1)
        for query_index in range(1, len(QUERY_PAIRS) + 1)
    ], dtype=np.float64)

    ratio_q1, ratio_median, ratio_q3 = np.percentile(
        paired_ratios, [25, 50, 75]
    )

    # 对每个查询先取 5 轮中位数，再统计 Python 较慢的查询数。
    python_slower_queries = 0
    for query_index in range(1, len(QUERY_PAIRS) + 1):
        py_query_median = np.median([
            TIMING_LOOKUP[
                (case_name, "Python", round_index, query_index)
            ]["outer_ns"]
            for round_index in range(1, N_ROUNDS + 1)
        ])
        cpp_query_median = np.median([
            TIMING_LOOKUP[
                (case_name, "C++", round_index, query_index)
            ]["outer_ns"]
            for round_index in range(1, N_ROUNDS + 1)
        ])
        python_slower_queries += int(py_query_median > cpp_query_median)

    SUMMARY_ROWS.append({
        "算法配置": case_name,
        "Python 外层中位数 (ms)": round(float(py_median), 3),
        "Python Q1–Q3 (ms)": f"{py_q1:.3f}–{py_q3:.3f}",
        "C++ 外层中位数 (ms)": round(float(cpp_median), 3),
        "C++ Q1–Q3 (ms)": f"{cpp_q1:.3f}–{cpp_q3:.3f}",
        "C++ 内核中位数 (ms)": round(kernel_median, 3),
        "配对耗时比 Python/C++": (
            f"{ratio_median:.2f} 倍（{ratio_q1:.2f}–{ratio_q3:.2f}）"
        ),
        f"{len(QUERY_PAIRS)} 个查询中 Python 较慢": (
            f"{python_slower_queries}/{len(QUERY_PAIRS)}"
        ),
        "Python 平均展开节点": round(float(np.mean([
            row["expanded_nodes"] for row in py_rows
        ])), 2),
        "C++ 平均展开节点": round(float(np.mean([
            row["expanded_nodes"] for row in cpp_rows
        ])), 2),
        "C++ 平均扫描边数": round(float(np.mean([
            row["scanned_edges"] for row in cpp_rows
        ])), 2),
        "C++ 平均启发式计算次数": round(float(np.mean([
            row["heuristic_evaluations"] for row in cpp_rows
        ])), 2),
    })

TIMING_DF = pd.DataFrame(TIMING_RECORDS)
SUMMARY_DF = pd.DataFrame(SUMMARY_ROWS)

# 每次运行单独建目录，避免覆盖此前实验结果。
RUN_TAG = time.strftime("%Y%m%d_%H%M%S") + f"_{time.perf_counter_ns() % 1_000_000:06d}"
RUN_OUTPUT_DIR = OUTPUT_DIR / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

RAW_RESULTS_FILE = RUN_OUTPUT_DIR / "astar_timing_raw.csv"
SUMMARY_RESULTS_FILE = RUN_OUTPUT_DIR / "astar_timing_summary.csv"
METADATA_FILE = RUN_OUTPUT_DIR / "astar_experiment_metadata.json"

TIMING_DF.to_csv(RAW_RESULTS_FILE, index=False, encoding="utf-8-sig")
SUMMARY_DF.to_csv(SUMMARY_RESULTS_FILE, index=False, encoding="utf-8-sig")

EXPERIMENT_METADATA = {
    "run_tag": RUN_TAG,
    "graph_policy": GRAPH_POLICY,
    "graph_sha256": GRAPH_SHA256,
    "query_sha256": QUERY_SHA256,
    "source_sha256": SOURCE_SHA256,
    "cpp_dll_sha256": CPP_DLL_SHA256,
    "environment": ENVIRONMENT,
    "nodes": N_NODES,
    "directed_edges": N_EDGES,
    "queries": len(QUERY_PAIRS),
    "rounds": N_ROUNDS,
    "warmup_queries_per_configuration_and_language": WARMUP_QUERY_COUNT,
    "random_seed": RANDOM_SEED,
    "case_order": list(CASE_ORDER),
    "landmark_ids": [int(node) for node in LANDMARK_IDS],
    "h0_scale_python": float(H0_SCALE),
    "h0_scale_cpp": float(CPP_H0_SCALE.value),
    "timing_method": {
        "outer": (
            "Python perf_counter_ns around the full adapter call; "
            "includes Python result construction and, for C++, ctypes/output conversion"
        ),
        "cpp_kernel": (
            "C++ astar_run internal steady_clock duration; "
            "excludes Python ctypes/output conversion"
        ),
        "paired_order": (
            "Configuration-language order shuffled deterministically "
            "within each query and round"
        ),
        "quartiles": "Q1/Q3 computed with numpy.percentile default linear interpolation",
    },
    "correctness_checked_runs": CHECKED_RUNS,
    "python_preprocessing_ns": {
        key: int(value) for key, value in PY_PREP_NS.items()
    },
    "cpp_preprocessing_kernel_ns": {
        "geometry": int(CPP_GEOMETRY_KERNEL_NS),
        "reverse_graph": int(CPP_REVERSE_KERNEL_NS),
        "alt": int(CPP_ALT_KERNEL_NS),
    },
    "output_files": {
        "raw_csv": RAW_RESULTS_FILE.name,
        "summary_csv": SUMMARY_RESULTS_FILE.name,
    },
}

with METADATA_FILE.open("w", encoding="utf-8") as file:
    json.dump(EXPERIMENT_METADATA, file, ensure_ascii=False, indent=2)

print(
    f"正式计时完成：{N_ROUNDS} 轮 × {len(QUERY_PAIRS)} 个查询 × "
    f"{len(CASE_ORDER)} 个配置 × 2 种语言 = {len(TIMING_RECORDS):,} 条记录"
)
print("说明：主比较使用外层耗时；C++ 内核耗时单独列出，不与 Python 外层耗时混比。")
display(SUMMARY_DF)
print(f"原始计时：{RAW_RESULTS_FILE}")
print(f"汇总表：{SUMMARY_RESULTS_FILE}")
print(f"实验元数据：{METADATA_FILE}")

正式计时完成：5 轮 × 100 个查询 × 15 个配置 × 2 种语言 = 15,000 条记录
说明：主比较使用外层耗时；C++ 内核耗时单独列出，不与 Python 外层耗时混比。


,算法配置,Python 外层中位数 (ms),Python Q1–Q3 (ms),C++ 外层中位数 (ms),C++ Q1–Q3 (ms),C++ 内核中位数 (ms),配对耗时比 Python/C++,100 个查询中 Python 较慢,Python 平均展开节点,C++ 平均展开节点,C++ 平均扫描边数,C++ 平均启发式计算次数
0,L0,359.046,330.371–407.738,10.662,8.317–17.322,10.558,35.02 倍（24.35–40.85）,100/100,8292.01,8292.01,18904.01,9581.90
1,L0-CachedScale,10.887,0.876–43.555,1.802,0.157–7.917,1.721,5.70 倍（4.64–6.44）,99/100,8292.01,8292.01,18904.01,9581.90
2,L1,9.550,0.776–42.874,1.628,0.160–7.369,1.549,5.09 倍（4.09–5.83）,99/100,8292.01,8292.01,18904.01,9581.90
3,L2,9.718,0.966–38.586,1.223,0.194–4.563,1.143,7.20 倍（4.76–8.24）,100/100,8292.01,8292.01,18904.01,9581.90
4,L3,9.549,1.052–36.456,0.876,0.221–2.504,0.804,10.95 倍（5.03–14.21）,100/100,8292.01,8292.01,18904.01,9581.90
5,L4,15.137,1.459–58.947,0.811,0.222–2.390,0.719,19.59 倍（6.49–25.30）,100/100,8292.01,8292.01,18904.01,9581.90
6,L5,15.426,1.553–57.102,1.028,0.243–3.567,0.960,14.48 倍（6.42–17.08）,100/100,8292.01,8292.01,18904.01,8523.40
7,L4-PLANE,12.131,1.256–43.991,0.661,0.213–1.935,0.569,18.42 倍（5.57–24.78）,100/100,8499.09,8499.09,19368.91,9808.33
8,L4-ALT4,8.659,1.140–42.359,0.518,0.201–1.529,0.451,19.03 倍（6.00–27.43）,100/100,3751.68,3751.68,8600.65,4535.77
9,L4-ALT8,9.151,1.423–40.269,0.517,0.211–1.362,0.439,19.85 倍（6.30–30.41）,100/100,2719.91,2719.91,6271.99,3327.60


原始计时：D:\pycharm\projects\route_plan\A3_临时工作区\06_CHATGPT\基础代码与优化测试结果\实验输出\astar_python_cpp\20261017_162040_819200\astar_timing_raw.csv
汇总表：D:\pycharm\projects\route_plan\A3_临时工作区\06_CHATGPT\基础代码与优化测试结果\实验输出\astar_python_cpp\20261017_162040_819200\astar_timing_summary.csv
实验元数据：D:\pycharm\projects\route_plan\A3_临时工作区\06_CHATGPT\基础代码与优化测试结果\实验输出\astar_python_cpp\20261017_162040_819200\astar_experiment_metadata.json


In [14]:
# Cell 7：分层收益拆解
# 正值含义：
# - 耗时变化为正：比参照层更慢
# - 展开节点变化为正：比参照层展开更多节点
# - 启发式提速倍数 > 1：候选启发式更快
# - 展开节点减少比例为正：候选启发式展开更少节点

SUMMARY_INDEX = SUMMARY_DF.set_index("算法配置")

LAYER_PREVIOUS = {
    "L0": None,
    "L0-CachedScale": "L0",
    "L1": "L0-CachedScale",
    "L2": "L1",
    "L3": "L2",
    "L4": "L3",
    "L5": "L4",
    "B1-H0": "L4",
}

LAYER_CASES = list(LAYER_PREVIOUS)

def summary_metric(case_name, language, column_suffix):
    if language == "Python":
        column = f"Python {column_suffix}"
    else:
        column = f"C++ {column_suffix}"
    return float(SUMMARY_INDEX.loc[case_name, column])


LAYER_ROWS = []

for language in ("Python", "C++"):
    baseline_ms = summary_metric("L0", language, "外层中位数 (ms)")
    baseline_expanded = summary_metric("L0", language, "平均展开节点")

    for case_name in LAYER_CASES:
        previous_case = LAYER_PREVIOUS[case_name]
        current_ms = summary_metric(case_name, language, "外层中位数 (ms)")
        current_expanded = summary_metric(case_name, language, "平均展开节点")

        if previous_case is None:
            previous_time_change = None
            previous_expanded_change = None
        else:
            previous_ms = summary_metric(
                previous_case, language, "外层中位数 (ms)"
            )
            previous_expanded = summary_metric(
                previous_case, language, "平均展开节点"
            )
            previous_time_change = 100.0 * (current_ms / previous_ms - 1.0)
            previous_expanded_change = (
                100.0 * (current_expanded / previous_expanded - 1.0)
                if previous_expanded else None
            )

        LAYER_ROWS.append({
            "语言": language,
            "算法层": case_name,
            "对比基准": previous_case or "自身基线",
            "外层中位耗时 (ms)": round(current_ms, 3),
            "相对对比基准耗时变化 (%)": (
                None if previous_time_change is None
                else round(previous_time_change, 2)
            ),
            "平均展开节点": round(current_expanded, 2),
            "相对对比基准展开变化 (%)": (
                None if previous_expanded_change is None
                else round(previous_expanded_change, 2)
            ),
            "相对 L0 提速倍数": round(baseline_ms / current_ms, 3),
            "相对 L0 展开节点变化 (%)": round(
                100.0 * (current_expanded / baseline_expanded - 1.0), 2
            ),
        })

LAYER_ANALYSIS_DF = pd.DataFrame(LAYER_ROWS)


# 固定 L4，只比较启发式变化；每个速度比都按同查询、同轮配对计算。
HEURISTIC_CASES = [
    "L4",
    "L4-PLANE",
    "L4-ALT4",
    "L4-ALT8",
    "L4-ALT16",
    "L4-MAX4",
    "L4-MAX8",
    "L4-MAX16",
]


def paired_speedup_summary(reference_case, candidate_case, language):
    ratios = np.asarray([
        TIMING_LOOKUP[
            (reference_case, language, round_index, query_index)
        ]["outer_ns"]
        / TIMING_LOOKUP[
            (candidate_case, language, round_index, query_index)
        ]["outer_ns"]
        for round_index in range(1, N_ROUNDS + 1)
        for query_index in range(1, len(QUERY_PAIRS) + 1)
    ], dtype=np.float64)

    q1, median, q3 = np.percentile(ratios, [25, 50, 75])
    return float(q1), float(median), float(q3)


HEURISTIC_ROWS = []

for language in ("Python", "C++"):
    l4_expanded = summary_metric("L4", language, "平均展开节点")

    for case_name in HEURISTIC_CASES:
        case_expanded = summary_metric(
            case_name, language, "平均展开节点"
        )
        ratio_q1, ratio_median, ratio_q3 = paired_speedup_summary(
            "L4", case_name, language
        )

        HEURISTIC_ROWS.append({
            "语言": language,
            "L4 启发式配置": case_name,
            "外层中位耗时 (ms)": round(
                summary_metric(case_name, language, "外层中位数 (ms)"), 3
            ),
            "相对 L4 配对提速倍数": (
                f"{ratio_median:.3f}（{ratio_q1:.3f}–{ratio_q3:.3f}）"
            ),
            "平均展开节点": round(case_expanded, 2),
            "相对 L4 展开节点减少 (%)": round(
                100.0 * (1.0 - case_expanded / l4_expanded), 2
            ),
        })

HEURISTIC_ANALYSIS_DF = pd.DataFrame(HEURISTIC_ROWS)

display(LAYER_ANALYSIS_DF)
display(HEURISTIC_ANALYSIS_DF)

LAYER_ANALYSIS_FILE = RUN_OUTPUT_DIR / "astar_layer_analysis.csv"
HEURISTIC_ANALYSIS_FILE = RUN_OUTPUT_DIR / "astar_heuristic_analysis.csv"

LAYER_ANALYSIS_DF.to_csv(
    LAYER_ANALYSIS_FILE, index=False, encoding="utf-8-sig"
)
HEURISTIC_ANALYSIS_DF.to_csv(
    HEURISTIC_ANALYSIS_FILE, index=False, encoding="utf-8-sig"
)

print("分层分析表已保存：")
print(LAYER_ANALYSIS_FILE)
print(HEURISTIC_ANALYSIS_FILE)
print(
    "解读提示：L4 启发式比较中，配对提速倍数大于 1 表示比 L4-H0 快；"
    "展开节点减少比例大于 0 表示搜索节点更少。"
)

,语言,算法层,对比基准,外层中位耗时 (ms),相对对比基准耗时变化 (%),平均展开节点,相对对比基准展开变化 (%),相对 L0 提速倍数,相对 L0 展开节点变化 (%)
0,Python,L0,自身基线,359.046,NaN,8292.01,NaN,1.000,0.00
1,Python,L0-CachedScale,L0,10.887,-96.97,8292.01,0.00,32.979,0.00
2,Python,L1,L0-CachedScale,9.550,-12.28,8292.01,0.00,37.596,0.00
3,Python,L2,L1,9.718,1.76,8292.01,0.00,36.946,0.00
4,Python,L3,L2,9.549,-1.74,8292.01,0.00,37.600,0.00
5,Python,L4,L3,15.137,58.52,8292.01,0.00,23.720,0.00
6,Python,L5,L4,15.426,1.91,8292.01,0.00,23.275,0.00
7,Python,B1-H0,L4,152.041,904.43,36677.27,342.32,2.362,342.32
8,C++,L0,自身基线,10.662,NaN,8292.01,NaN,1.000,0.00
9,C++,L0-CachedScale,L0,1.802,-83.10,8292.01,0.00,5.917,0.00


,语言,L4 启发式配置,外层中位耗时 (ms),相对 L4 配对提速倍数,平均展开节点,相对 L4 展开节点减少 (%)
0,Python,L4,15.137,1.000（1.000–1.000）,8292.01,0.00
1,Python,L4-PLANE,12.131,1.263（1.168–1.324）,8499.09,-2.50
2,Python,L4-ALT4,8.659,1.261（0.972–1.825）,3751.68,54.76
3,Python,L4-ALT8,9.151,1.143（0.896–1.857）,2719.91,67.20
4,Python,L4-ALT16,10.541,0.920（0.695–1.807）,1856.52,77.61
5,Python,L4-MAX4,8.863,1.079（0.895–1.512）,3307.20,60.12
6,Python,L4-MAX8,10.305,1.012（0.787–1.610）,2450.85,70.44
7,Python,L4-MAX16,11.813,0.850（0.658–1.571）,1807.35,78.20
8,C++,L4,0.811,1.000（1.000–1.000）,8292.01,0.00
9,C++,L4-PLANE,0.661,1.170（0.996–1.342）,8499.09,-2.50


分层分析表已保存：
D:\pycharm\projects\route_plan\A3_临时工作区\06_CHATGPT\基础代码与优化测试结果\实验输出\astar_python_cpp\20261017_162040_819200\astar_layer_analysis.csv
D:\pycharm\projects\route_plan\A3_临时工作区\06_CHATGPT\基础代码与优化测试结果\实验输出\astar_python_cpp\20261017_162040_819200\astar_heuristic_analysis.csv
解读提示：L4 启发式比较中，配对提速倍数大于 1 表示比 L4-H0 快；展开节点减少比例大于 0 表示搜索节点更少。
